# Fair test of evidence expansion

Upload **phase4_expansion_source.zip** to My Drive root, choose an A100, and Run all.
Return **phase4_expansion_results.zip** for audit. Packages use uv; source stays in a separate
checksummed Drive ZIP. No long encoded source cell.

**30 newly authored multi-block questions**: six each Requests/Express/Flask, twelve Click
(new repository). **120 frozen regression questions** are also run, unchanged.
All fresh proofs require multiple indexed chunks. Labels and success rule were frozen/hashed
before new retrieval; no git commit was created because the user forbids commits.

Three fixed policies independently traverse production search: **plain rerank20**, **rerank20
+ callee**, **rerank20 + both**. First-stage depth 50/signal, returned top10; expansion keeps
5 hits and takes at most 2 additions from each of 3 seeds. No parameter tuning.
One resident scorer shares exact pair scores across policies with identical inputs:
**quality only, no latency/speedup measurement**. 150 questions, 450 policy outputs.

Primary callee must improve new-question complete proof by at least +0.05 with a paired 95%
lower bound above zero, lose no more than 0.02 MRR or proof on any old stratum, and grow
returned text by no more than 10% separately on fresh30 and regression120. Both is secondary.
The analyzer validates hashes, candidates, expansion reconstruction, metrics and text budgets.
On rejection, leave expansion off and move to Phase 5; do not iterate on these questions.

Name matching is not a call graph. Independent label review and answer-generation evaluation
remain pending. A100 is experiment-only; deployment undecided. Expansion stays off by default,
and it never runs on unreranked hybrid/CPU searches. No default is changed automatically.

In [ ]:
SOURCE_ZIP = "/content/drive/MyDrive/phase4_expansion_source.zip"

In [ ]:
# Load the separate source ZIP that you uploaded to Google Drive.
import hashlib, pathlib, zipfile
from google.colab import drive

drive.mount("/content/drive")
source_zip = pathlib.Path(SOURCE_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(
        f"Upload phase4_expansion_source.zip to My Drive, or update SOURCE_ZIP. Missing: {source_zip}"
    )
expected_sha256 = "adfdea0fede7412ad77dec0955487bfe4e26cf128fae2ac9cf06eca8bf7e3c9c"
if hashlib.sha256(source_zip.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError(
        "Source ZIP does not match this notebook. Download the matching ZIP and notebook again."
    )
PAYLOAD = pathlib.Path("/content/phase4_expansion_payload")
with zipfile.ZipFile(source_zip) as package:
    package.extractall(PAYLOAD)
print("Loaded project code and question datasets from", source_zip.name)

In [ ]:
import importlib.util, subprocess, sys
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--system',
                '--python', sys.executable, '-q', 'chromadb==1.5.9', 'fastapi==0.162.0', 'gitpython==3.2.1', 'langchain-chroma==1.1.0', 'langchain-core==1.6.13', 'langchain-text-splitters==1.1.3', 'langgraph==1.2.19', 'pydantic==2.14.1', 'pydantic-settings==2.16.1', 'rank-bm25==0.2.2', 'rich==15.0.0', 'sentence-transformers==5.7.0', 'transformers==4.57.6', 'tree-sitter==0.26.0', 'tree-sitter-language-pack==1.22.1', 'typer==0.27.3', 'uvicorn==0.54.0', 'astchunk==0.1.0'], check=True)

In [ ]:
import json, os, pathlib, shutil, subprocess, sys
REPOS = pathlib.Path('/content/expansion_repos')
RESULTS = pathlib.Path('/content/expansion_results')
LABELS = PAYLOAD / 'eval/datasets/phase4_expansion_v1'
if RESULTS.exists():
    raise FileExistsError('Results already exist. Download them before starting a new runtime.')
REPOS.mkdir(exist_ok=True)
RESULTS.mkdir()
shutil.copytree(LABELS, RESULTS / 'labels')
shutil.copyfile(source_zip, RESULTS / 'phase4_expansion_source.zip')
env = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(PAYLOAD), str(PAYLOAD / 'src')]),
       'TOKENIZERS_PARALLELISM': 'false'}
manifest = json.loads((LABELS / 'manifest.json').read_text())
repos = {e['repo']: e['commit_sha'] for e in manifest['datasets']}
for repo, sha in repos.items():
    checkout = REPOS / repo.replace('/', '__')
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                        'https://github.com/' + repo, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', sha], check=True)
    subprocess.run([sys.executable, str(PAYLOAD / 'eval/run_expansion_eval.py'),
                    '--labels', str(LABELS), '--repo', repo, '--checkout', str(checkout),
                    '--output', str(RESULTS / repo.replace('/', '__')),
                    '--mode', 'full', '--device', 'cuda'], env=env, check=True)
subprocess.run([sys.executable, str(PAYLOAD / 'eval/analyze_expansion_run.py'), str(RESULTS)],
                env=env, check=True)
print((RESULTS / 'aggregate/summary.md').read_text())

In [ ]:
from google.colab import files
archive = pathlib.Path('/content/phase4_expansion_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.rglob('*')):
        if path.is_file() and 'isolated_index' not in path.parts:
            bundle.write(path, path.relative_to(RESULTS))
files.download(str(archive))